# High-Density Object Segmentation — full pipeline on a Colab T4

Runs the **whole** experiment end to end: dense subset → fine-tune YOLOv8s-seg →
the unified evaluation of all four methods → figures → generated README table.

Everything happens on one machine in one session, which is the point. The
previous version of this project trained in one place and evaluated in another
and then printed both sets of numbers as a single table.

**Before you run:** `Runtime → Change runtime type → T4 GPU`. Then `Runtime → Run all`.
Takes roughly 15 minutes, most of it the COCO download.


## 1 — Confirm we actually have a GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again."
print("torch", torch.__version__, "| cuda", torch.version.cuda)

## 2 — Get the code

In [ ]:
!git clone -q https://github.com/Zakuroooo/high-density-object-segmentation.git
%cd high-density-object-segmentation
!git log --oneline -1

## 3 — Dependencies

In [ ]:
!pip install -q ultralytics==8.4.38 pycocotools==2.0.10 scikit-image nbformat nbconvert 2>&1 | tail -2
import ultralytics; print("ultralytics", ultralytics.__version__)

## 4 — COCO val2017

Images and instance annotations only. The train2017 annotation files are ~1.5 GB
and nothing here reads them, so they are deleted straight after unzipping.

In [ ]:
%%bash
mkdir -p data && cd data
[ -f images/val2017/000000000139.jpg ] || {
  curl -sL -O http://images.cocodataset.org/zips/val2017.zip
  curl -sL -O http://images.cocodataset.org/annotations/annotations_trainval2017.zip
  unzip -q annotations_trainval2017.zip -d annotations
  mkdir -p images && unzip -q val2017.zip -d images
  rm -f val2017.zip annotations_trainval2017.zip
  cd annotations/annotations && rm -f captions_*.json instances_train2017.json person_keypoints_*.json
}
echo "images: $(ls data/images/val2017 2>/dev/null | wc -l)" 

## 5 — Build the dense subset

A seeded uniform sample from every COCO val2017 image with 5-50 non-crowd
objects, split 400/50/50. The seed lives in `src/config.py`, so this cell is
reproducible and so is every number downstream of it.

In [ ]:
!PYTHONPATH=src python src/prepare_yolo_data.py 2>&1 | tail -12
import json
s = json.load(open("results/metrics/data_split.json"))
print("\npool:", s["total_dense_images"], "| seed:", s["seed"], "| sampling:", s["sampling"])
print("split:", {k: len(s[k]) for k in ("train","val","test")})
print("train/test overlap:", set(s["train"]) & set(s["test"]) or "none")

## 6 — Fine-tune YOLOv8s-seg

10 epochs at 640px on the 400-image dense train split. On a T4 this is a few
minutes; on an 8 GB M2 the same run took ~10 minutes *per epoch* and swapped.

In [ ]:
!PYTHONPATH=src python src/train.py 2>&1 | grep -vE "^\s*$" | tail -30

In [ ]:
import pandas as pd
r = pd.read_csv("runs/dense-seg/results.csv")
r.columns = [c.strip() for c in r.columns]
print(r[[c for c in r.columns if "epoch" in c or "mAP50" in c or "loss" in c.lower()][:6]].tail(5).to_string(index=False))

## 7 — The unified evaluation

Watershed, KMeans, YOLOv8s-seg and the hybrid, scored on the **same** 50
held-out test images with the **same** metric definitions in **one** process.
This single output file is what every table and figure downstream is built from.

In [ ]:
!PYTHONPATH=src python src/evaluate.py 2>&1 | tail -20

## 8 — Figures and the generated README table

In [ ]:
!PYTHONPATH=src python src/figures.py 2>&1 | tail -10
!python scripts/update_readme.py
!python scripts/update_readme.py --check

In [ ]:
from IPython.display import Image, display
for f in ["comparison.png", "count_scatter.png", "inference_time.png", "density_gate.png"]:
    display(Image(filename=f"results/figures/{f}"))

## 9 — Generate and execute the notebooks (so they ship with outputs)

In [ ]:
!python scripts/build_notebooks.py
!cd notebooks && PYTHONPATH=../src jupyter nbconvert --to notebook --execute --inplace \
    01_eda.ipynb 02_methods_and_results.ipynb 2>&1 | tail -5
import nbformat
for n in ["01_eda", "02_methods_and_results"]:
    nb = nbformat.read(f"notebooks/{n}.ipynb", as_version=4)
    ex = sum(1 for c in nb.cells if c.get("execution_count"))
    print(f"{n}: {ex} executed cells")

## 10 — The results

Read this table before downloading. These are the numbers that will go into the
README, the report and the resume.

In [ ]:
import json
res = json.load(open("results/metrics/unified_results.json"))
p = res["provenance"]
print(f"v{p['experiment_version']}  {p['date']}  seed={p['seed']}  device={p['device']}")
print(f"test images={p['n_test_images']}  matching={p['iou_match_mode']}")
if p.get("training"): print("training:", p["training"])
print()
print(f"{'method':<26}{'IoU':>8}{'acc%':>8}{'MAE':>8}{'ms':>9}")
print("-"*59)
for k in ["watershed","kmeans","yolo","hybrid"]:
    s = res["summary"][k]
    print(f"{k:<26}{s['mean_iou']:>8.4f}{s['count_accuracy_pct']:>8.1f}{s['mae']:>8.2f}{s['median_inference_ms']:>9.1f}")
h = res["summary"]["hybrid"]
print(f"\nhybrid gate fired: {h['dense_gate_fired']}/{p['n_test_images']}   blend applied: {h['watershed_blend_applied']}")

## 11 — Download

One zip with the weights, the results file, the figures, the executed notebooks
and the updated README. Unzip it over your local clone, then commit.

In [ ]:
!mkdir -p /content/out && cd /content/high-density-object-segmentation && \
 zip -qr /content/out/hdos_results.zip \
   weights/ results/ notebooks/01_eda.ipynb notebooks/02_methods_and_results.ipynb \
   README.md runs/dense-seg/results.csv runs/dense-seg/results.png
!ls -lh /content/out/hdos_results.zip
from google.colab import files
files.download("/content/out/hdos_results.zip")